In [ ]:
#| default_exp graph

# Project graph

> Hybrid project graph across nbdev notebooks and handwritten Python: architecture layers, hubs, communities, blast radius, and dependency paths.

Flat notebook lists break down once a project grows to dozens of notebooks and tests. This module builds a deterministic, pure-Python graph in memory (no stale on-disk index) linking notebooks, export targets, handwritten `.py` files, symbols, `@patch` methods, `@delegates`, inheritance, Markdown references, and tests.

In [ ]:
#| export
import ast, difflib, json, re
from collections import Counter, defaultdict, deque
from dataclasses import dataclass, field
from pathlib import Path

from fastcore.nbio import read_nb

from nbskill.core import (cell_defs, cell_directives, cell_export_kind, cell_export_module, cell_imports,
    cell_source, chapters, handwritten_py_paths, is_exported, notebook_export_targets, notebook_paths,
    package_name, parse_cell_ast, project_root)

In [ ]:
from tempfile import TemporaryDirectory
from fastcore.nbio import mk_cell, new_nb, write_nb
from fastcore.test import test_eq, test_fail

## Graph data model

Three lightweight records describe the project: `ModuleNode` for each notebook or handwritten Python file (never duplicating generated `.py` modules), `SymbolNode` for every top-level definition or `@patch` method, and `GraphEdge` for typed relationships tagged with provenance confidence (`exact`, `resolved`, or `ambiguous`).

In [ ]:
#| export
_BACKTICK_REF_RE = re.compile(r"`([A-Za-z_][A-Za-z0-9_.]*)`")
_NUM_PREFIX_RE = re.compile(r"^(\d+)[a-z]?_")
_COMMON_ATTRS = frozenset(("get set items keys values append extend insert pop remove clear copy update add discard union sort reverse count index join split "
    "rsplit splitlines strip lstrip rstrip replace format lower upper startswith endswith encode decode read write close open exists is_file is_dir mkdir "
    "resolve relative_to match search findall sub group groups read_text write_text").split())

@dataclass
class ModuleNode:
    "One notebook or handwritten Python file in the project graph."
    id: str
    abs_path: str
    role: str  # 'library', 'unexported_nb', 'test', 'script', or 'handwritten_py'
    title: str = ""
    subtitle: str = ""
    export_modules: list[str] = field(default_factory=list)
    chapters: list[str] = field(default_factory=list)
    cell_count: int = 0
    layer: int = 0
    community: int = 0
    centrality: float = 0.0
    in_degree: int = 0
    out_degree: int = 0

@dataclass
class SymbolNode:
    "One function, class, or `@patch` method in a notebook or handwritten Python file."
    uid: str
    name: str
    qualname: str
    kind: str  # 'function', 'async_function', 'class', or 'patch'
    node_id: str
    module: str
    visibility: str  # 'public', 'exporti', 'private', 'lesson', or 'py'
    chapter: str = ""
    cell_id: str = ""
    lineno: int = 0
    receiver: str = ""
    bases: tuple[str, ...] = ()
    delegates_to: str = ""

In [ ]:
#| export
@dataclass(frozen=True)
class GraphEdge:
    "One typed, directed edge with explicit confidence."
    src: str
    dst: str
    kind: str  # 'imports', 'calls', 'patches', 'inherits', 'delegates', 'docs_ref', or 'tests'
    confidence: str  # 'exact', 'resolved', or 'ambiguous'
    path: str = ""
    cell_id: str = ""
    chapter: str = ""
    lineno: int = 0

@dataclass
class Community:
    "One cohesive subsystem cluster discovered from module coupling."
    id: int
    name: str
    nodes: list[str] = field(default_factory=list)
    hub: str = ""
    key_symbols: list[str] = field(default_factory=list)
    internal_edges: int = 0
    external_edges: int = 0

`ProjectGraph` holds the indexed nodes, edges, communities, cycles, and serialization helpers (`to_json` and `to_mermaid`).

In [ ]:
#| export
def _mermaid_id(raw: str) -> str: return "n_" + re.sub(r"[^A-Za-z0-9_]", "_", raw)

@dataclass
class ProjectGraph:
    "In-memory structural graph of an nbdev project and its handwritten Python files."
    root: str
    package: str | None
    modules: dict = field(default_factory=dict)
    symbols: dict = field(default_factory=dict)
    edges: list[GraphEdge] = field(default_factory=list)
    mod_to_nodes: dict = field(default_factory=dict)
    mod_exports: dict = field(default_factory=dict)
    cycles: list = field(default_factory=list)
    communities: list[Community] = field(default_factory=list)
    surprising: list = field(default_factory=list)
    doc_refs: dict = field(default_factory=dict)

    def symbols_named(self, query: str) -> list[SymbolNode]:
        "Return all `SymbolNode` entries matching `query` (`name` or `Receiver.name`)."
        if "." in query:
            recv, nm = query.split(".", 1)
            return [s for s in self.symbols.values() if s.name == nm and s.receiver == recv]
        return [s for s in self.symbols.values() if s.name == query or s.qualname == query]

    def patches_for(self, class_name: str) -> list[SymbolNode]:
        "Return all `@patch` methods targeting `class_name` across the project."
        return [s for s in self.symbols.values() if s.kind == "patch" and s.receiver == class_name]

    def edges_from(self, src: str, *, kinds: set[str] | None = None) -> list[GraphEdge]:
        return [e for e in self.edges if e.src == src and (kinds is None or e.kind in kinds)]

    def edges_to(self, dst: str, *, kinds: set[str] | None = None) -> list[GraphEdge]:
        return [e for e in self.edges if e.dst == dst and (kinds is None or e.kind in kinds)]

In [ ]:
#| export
def _graph_to_dict(g: ProjectGraph) -> dict:
    mods = {k: dict(role=m.role, title=m.title, export_modules=m.export_modules, layer=m.layer, community=m.community,
        centrality=round(m.centrality, 4), in_degree=m.in_degree, out_degree=m.out_degree) for k, m in g.modules.items()}
    syms = {k: dict(name=s.name, qualname=s.qualname, kind=s.kind, node_id=s.node_id, module=s.module, visibility=s.visibility,
        chapter=s.chapter, cell_id=s.cell_id, lineno=s.lineno, receiver=s.receiver, bases=list(s.bases),
        delegates_to=s.delegates_to) for k, s in g.symbols.items()}
    edges = [dict(src=e.src, dst=e.dst, kind=e.kind, confidence=e.confidence, path=e.path, cell_id=e.cell_id, lineno=e.lineno) for e in g.edges]
    comms = [dict(id=c.id, name=c.name, hub=c.hub, nodes=c.nodes, key_symbols=c.key_symbols) for c in g.communities]
    surp = [dict(src=a, dst=b, reason=r) for a, b, r in g.surprising]
    return dict(root=g.root, package=g.package, modules=mods, symbols=syms, edges=edges, cycles=g.cycles, communities=comms, surprising=surp)

def _graph_to_json(g: ProjectGraph, indent: int = 2) -> str: return json.dumps(_graph_to_dict(g), indent=indent)

def _graph_to_mermaid(g: ProjectGraph, level: str = "module", *, include_tests: bool = False) -> str:
    lines, seen = ["flowchart LR"], set()
    if level == "community":
        for c in g.communities: lines.append(f'    c_{c.id}["{c.name} ({len(c.nodes)} modules)"]'.replace('""', '"'))
        for e in g.edges:
            if e.kind != "imports": continue
            ma, mb = g.modules.get(e.src), g.modules.get(e.dst)
            if not ma or not mb or ma.community == mb.community or (not include_tests and (ma.role == "test" or mb.role == "test")): continue
            pair = (ma.community, mb.community)
            if pair not in seen:
                seen.add(pair)
                lines.append(f"    c_{pair[0]} --> c_{pair[1]}")
        return "\n".join(lines)
    shown = {nid for nid, m in g.modules.items() if include_tests or m.role != "test"}
    for nid in sorted(shown): lines.append(f'    {_mermaid_id(nid)}["{Path(nid).name} (L{g.modules[nid].layer})"]')
    for e in g.edges:
        if e.kind != "imports" or e.src not in shown or e.dst not in shown or (e.src, e.dst) in seen: continue
        seen.add((e.src, e.dst))
        lines.append(f"    {_mermaid_id(e.src)} --> {_mermaid_id(e.dst)}")
    return "\n".join(lines)

ProjectGraph.to_dict, ProjectGraph.to_json, ProjectGraph.to_mermaid = _graph_to_dict, _graph_to_json, _graph_to_mermaid

Serializing a minimal `ProjectGraph` confirms both Mermaid flowchart generation and JSON output.

In [ ]:
m_core = ModuleNode("nbs/00_core.ipynb", "/tmp/demo/nbs/00_core.ipynb", "library", "Core", layer=0)
m_app = ModuleNode("nbs/01_app.ipynb", "/tmp/demo/nbs/01_app.ipynb", "library", "App", layer=1)
g_demo = ProjectGraph("/tmp/demo", "demo", modules={"nbs/00_core.ipynb": m_core, "nbs/01_app.ipynb": m_app},
    edges=[GraphEdge("nbs/01_app.ipynb", "nbs/00_core.ipynb", "imports", "exact")])
mmd = g_demo.to_mermaid()
assert "flowchart LR" in mmd and "01_app.ipynb (L1)" in mmd and "-->" in mmd
test_eq(json.loads(g_demo.to_json())["package"], "demo")
mmd

## Building the hybrid project graph

Building the graph happens in two fast passes: first collect all notebook and handwritten Python nodes, their export modules, definitions, and Markdown backtick references; second resolve imports (including relative and internal star imports), calls, `@patch` links, inheritance, `@delegates`, and test coverage.

In [ ]:
#| export
_GRAPH_CACHE = {}

def _rel_str(p: Path, root: Path) -> str:
    try: return p.resolve().relative_to(root.resolve()).as_posix()
    except ValueError: return str(p)

def _nb_title_sub(nb) -> tuple[str, str]:
    title, sub = "", ""
    for c in nb.cells[:6]:
        if c.get("cell_type") != "markdown": continue
        for line in cell_source(c).splitlines():
            s = line.strip()
            if not title and s.startswith("# "): title = s[2:].strip()
            elif not sub and s.startswith(">"): sub = s[1:].strip()
    return title or "(untitled)", sub

def _py_role_and_mod(rel: str, pkg: str | None) -> tuple[str, str]:
    parts = Path(rel).parts
    if parts and (parts[0] in ("tests", "test") or Path(rel).name.startswith("test_")): return "test", ""
    if pkg and parts and parts[0] == pkg:
        stem_parts = list(parts[:-1]) + ([] if Path(rel).stem == "__init__" else [Path(rel).stem])
        return "handwritten_py", ".".join(stem_parts)
    if parts and parts[0] in ("scripts", "ops", "tools", "bin"): return "script", ""
    return ("script" if len(parts) == 1 else "handwritten_py"), ""

In [ ]:
#| export
def _qual_mod(pkg: str | None, mod: str) -> str:
    if not mod: return ""
    if not pkg or mod == pkg or mod.startswith(pkg + "."): return mod
    return f"{pkg}.{mod}"

def _resolve_import_mod(imp_mod: str, cur_mod: str) -> str:
    if not imp_mod.startswith("."): return imp_mod
    dots = len(imp_mod) - len(imp_mod.lstrip("."))
    tail = imp_mod[dots:]
    base = cur_mod.split(".")[:-dots] if cur_mod else []
    return ".".join(base + ([tail] if tail else []))

def _extract_reexports(tree: ast.Module) -> set[str]:
    names = set()
    for node in tree.body:
        if not isinstance(node, ast.Assign): continue
        if not any(isinstance(t, ast.Name) and t.id in ("_all_", "__all__") for t in node.targets): continue
        if isinstance(node.value, (ast.List, ast.Tuple, ast.Set)):
            for elt in node.value.elts:
                if isinstance(elt, ast.Name): names.add(elt.id)
                elif isinstance(elt, ast.Constant) and isinstance(elt.value, str): names.add(elt.value)
    return names

AST extractors read top-level definitions, imports, and call sites attributed to their enclosing function or class.

In [ ]:
#| export
def _py_defs_and_imports(tree: ast.Module):
    defs, imps = [], []
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            kind = "async_function" if isinstance(node, ast.AsyncFunctionDef) else "function"
            defs.append((node.name, kind, node.lineno, "", not node.name.startswith("_"), (), ""))
        elif isinstance(node, ast.ClassDef):
            defs.append((node.name, "class", node.lineno, "", not node.name.startswith("_"), tuple(ast.unparse(b) for b in node.bases), ""))
        elif isinstance(node, ast.Import):
            for a in node.names: imps.append((a.name, a.name, a.asname or a.name.split(".", 1)[0], node.lineno, False))
        elif isinstance(node, ast.ImportFrom):
            mod = "." * node.level + (node.module or "")
            for a in node.names: imps.append((mod, a.name, a.asname or a.name, node.lineno, a.name == "*"))
    return defs, imps

def _ast_calls(tree: ast.Module) -> list:
    "Return `(enclosing_def, base_or_empty, name, lineno)` for every Call node."
    out = []
    def _collect(node, owner: str):
        for child in ast.walk(node):
            if not isinstance(child, ast.Call): continue
            fn = child.func
            if isinstance(fn, ast.Name): out.append((owner, "", fn.id, child.lineno))
            elif isinstance(fn, ast.Attribute):
                base = fn.value.id if isinstance(fn.value, ast.Name) else ""
                out.append((owner, base, fn.attr, child.lineno))
    for stmt in tree.body:
        owner = stmt.name if isinstance(stmt, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) else ""
        _collect(stmt, owner)
    return out

Graph analytics run deterministically on the collected module graph: Tarjan strongly connected components find cycles, topological layering orders foundations before entrypoints, PageRank highlights God-Hubs, and inverse-degree label propagation groups large repos into cohesive subsystems.

In [ ]:
#| export
def _tarjan_sccs(nodes: list[str], adj: dict) -> list:
    index, stack, on_stack, idx_map, low, sccs = 0, [], set(), {}, {}, []
    def strong(v):
        nonlocal index
        idx_map[v] = low[v] = index
        index += 1
        stack.append(v)
        on_stack.add(v)
        for w in sorted(adj.get(v, ())):
            if w not in idx_map:
                strong(w)
                low[v] = min(low[v], low[w])
            elif w in on_stack: low[v] = min(low[v], idx_map[w])
        if low[v] == idx_map[v]:
            comp = []
            while True:
                w = stack.pop()
                on_stack.remove(w)
                comp.append(w)
                if w == v: break
            sccs.append(sorted(comp))
    for n in sorted(nodes):
        if n not in idx_map: strong(n)
    return sccs

def _compute_layers(nodes: list[str], adj: dict, sccs: list) -> dict:
    comp_of = {n: i for i, comp in enumerate(sccs) for n in comp}
    comp_adj = defaultdict(set)
    for u in nodes:
        for v in adj.get(u, ()):
            if comp_of[u] != comp_of[v]: comp_adj[comp_of[u]].add(comp_of[v])
    memo = {}
    def depth(ci):
        if ci in memo: return memo[ci]
        deps = comp_adj.get(ci, ())
        memo[ci] = 0 if not deps else 1 + max(depth(d) for d in deps)
        return memo[ci]
    return {n: depth(comp_of[n]) for n in nodes}

def _compute_pagerank(nodes: list[str], adj: dict, iters: int = 20, d: float = 0.85) -> dict:
    n = len(nodes)
    if not n: return {}
    scores = {u: 1.0 / n for u in nodes}
    for _ in range(iters):
        nxt = {u: (1.0 - d) / n for u in nodes}
        for u in nodes:
            targets = [v for v in adj.get(u, ()) if v in scores]
            share = d * scores[u] / (len(targets) if targets else n)
            for v in (targets or nodes): nxt[v] += share
        scores = nxt
    return scores

In [ ]:
#| export
def _community_weights(g: ProjectGraph, nb_nodes: list[str], imp_adj: dict) -> dict:
    nb_set, hub_cap = set(nb_nodes), max(6, int(0.25 * len(nb_nodes)))
    undir = defaultdict(lambda: defaultdict(float))
    for u in nb_nodes:
        for v in imp_adj.get(u, ()):
            if v not in nb_set or u == v or g.modules[v].in_degree > hub_cap: continue
            w = 2.0 / (1.0 + 0.25 * g.modules[v].in_degree)
            undir[u][v] += w
            undir[v][u] += w
    for mod_nodes in g.mod_to_nodes.values():
        shared = [n for n in mod_nodes if n in nb_set]
        for i, u in enumerate(shared):
            for v in shared[i + 1:]:
                undir[u][v] += 3.0
                undir[v][u] += 3.0
    by_dir = defaultdict(list)
    for u in nb_nodes:
        pu = Path(u).parent.as_posix()
        if pu not in (".", "nbs"): by_dir[pu].append(u)
    for peers in by_dir.values():
        for i, u in enumerate(peers):
            for v in peers[i + 1:]:
                undir[u][v] += 2.0
                undir[v][u] += 2.0
    return undir

Label propagation runs deterministically over those hub-damped weights to partition large repositories into named subsystem communities.

In [ ]:
#| export
def _compute_communities(g: ProjectGraph, nb_nodes: list[str], imp_adj: dict) -> list[Community]:
    if not nb_nodes: return []
    undir, labels = _community_weights(g, nb_nodes, imp_adj), {n: i for i, n in enumerate(sorted(nb_nodes))}
    for _ in range(20):
        changed = False
        for n in sorted(nb_nodes):
            nbrs = undir.get(n)
            if not nbrs: continue
            weights = defaultdict(float)
            for v, w in nbrs.items(): weights[labels[v]] += w
            best = sorted(weights.items(), key=lambda kv: (-round(kv[1], 6), kv[0]))[0][0]
            if best != labels[n]:
                labels[n] = best
                changed = True
        if not changed: break
    groups = defaultdict(list)
    for n, cid in labels.items(): groups[cid].append(n)
    comms = []
    for new_id, members in enumerate(sorted(groups.values(), key=lambda xs: (-len(xs), xs[0]))):
        mset = set(members)
        for n in members: g.modules[n].community = new_id
        hub = sorted(members, key=lambda n: (-g.modules[n].in_degree, -g.modules[n].centrality, n))[0]
        parents = {Path(n).parent.as_posix() for n in members}
        name = Path(next(iter(parents))).name if len(parents) == 1 and next(iter(parents)) not in (".", "nbs") else (g.modules[hub].title or Path(hub).stem)
        int_e = sum(1 for u in members for v in imp_adj.get(u, ()) if v in mset)
        ext_e = sum(1 for u in members for v in imp_adj.get(u, ()) if v not in mset)
        key_syms = [s.qualname for s in g.symbols.values() if s.node_id in mset and s.visibility == "public"][:6]
        comms.append(Community(new_id, name, sorted(members), hub, key_syms, int_e, ext_e))
    return comms

Surprising connections highlight import cycles and lower-numbered notebooks importing higher-numbered ones.

In [ ]:
#| export
def _nb_order_num(rel: str):
    m = _NUM_PREFIX_RE.match(Path(rel).name)
    return (Path(rel).parent.as_posix(), int(m.group(1))) if m else None

def _find_surprising(g: ProjectGraph, imp_adj: dict) -> list:
    out = [(cyc[0], cyc[1] if len(cyc) > 1 else cyc[0], f"import cycle: {' -> '.join(cyc + [cyc[0]])}") for cyc in g.cycles]
    for u in sorted(imp_adj):
        ou = _nb_order_num(u)
        for v in sorted(imp_adj[u]):
            ov = _nb_order_num(v)
            if ou and ov and ou[0] == ov[0] and ou[1] < ov[1]:
                out.append((u, v, f"layer inversion ({Path(u).name} imports higher-numbered {Path(v).name})"))
    return out

Indexing collects nodes and symbols from both notebooks and handwritten Python files, then resolves imports, calls, `@patch`, inheritance, `@delegates`, and Markdown references.

In [ ]:
#| export
def _cache_signature(root: Path, nb_files: list[Path], py_files: list[Path], include_py: bool) -> tuple:
    stats = []
    for f in (*nb_files, *py_files):
        try:
            st = f.stat()
            stats.append((f.as_posix(), st.st_mtime_ns, st.st_size))
        except OSError: pass
    return (root.resolve().as_posix(), include_py, tuple(stats))

def _resolve_scope(path: str | Path) -> tuple:
    p = Path(path).expanduser().resolve()
    root = project_root(p)
    nbs = notebook_paths(p)
    if p.is_dir() and p != root and (root / "nbs").is_dir() and p == (root / "nbs").resolve(): py_root = root
    else: py_root = p if p.is_dir() else root
    return root, nbs, handwritten_py_paths(py_root)

def _add_symbol(g: ProjectGraph, s: SymbolNode):
    if s.uid in g.symbols: s.uid = f"{s.uid}@{s.cell_id or s.lineno}"
    g.symbols[s.uid] = s
    if s.module: g.mod_exports.setdefault(s.module, {}).setdefault(s.name, []).append(s.uid)

In [ ]:
#| export
def _index_notebooks(g: ProjectGraph, root: Path, nb_files: list[Path]) -> dict:
    raw_cells = {}
    for np in nb_files:
        nb, rel = read_nb(np), _rel_str(np, root)
        title, sub = _nb_title_sub(nb)
        chaps = chapters(nb)
        ch_by_idx = {i: ch.title for ch in chaps for i in range(ch.start, ch.end)}
        targets = notebook_export_targets(np, nb)
        default_exp = next((cell_directives(c)["default_exp"] for c in nb.cells if "default_exp" in cell_directives(c)), None)
        exp_mods = sorted({_qual_mod(g.package, m) for m in targets})
        g.modules[rel] = ModuleNode(rel, str(np.resolve()), "library" if exp_mods else "unexported_nb", title, sub, exp_mods,
            [ch.title for ch in chaps if ch.title != "Notebook head" or ch.defs], len(nb.cells))
        for fm in exp_mods: g.mod_to_nodes.setdefault(fm, []).append(rel)
        raw_cells[rel] = (nb, ch_by_idx, default_exp)
        for idx, cell in enumerate(nb.cells):
            cid, ch = str(cell.get("id", f"cell-{idx}")), ch_by_idx.get(idx, "")
            if cell.get("cell_type") == "markdown":
                for ref in _BACKTICK_REF_RE.findall(cell_source(cell)): g.doc_refs.setdefault(rel, []).append((ref, cid, ch))
                continue
            if cell.get("cell_type") != "code": continue
            exp_kind = cell_export_kind(cell)
            exp_mod = _qual_mod(g.package, cell_export_module(cell, default_exp) or "")
            for d in cell_defs(cell):
                qname = f"{d.receiver}.{d.name}" if d.receiver else d.name
                vis = ("exporti" if exp_kind == "exporti" else ("public" if d.is_public else "private")) if exp_kind else "lesson"
                _add_symbol(g, SymbolNode(f"{rel}::{qname}", d.name, qname, d.kind, rel, exp_mod if exp_kind else "",
                    vis, ch, cid, d.lineno, d.receiver, d.bases, d.delegates_to))
    return raw_cells

def _index_py_files(g: ProjectGraph, root: Path, py_files: list[Path]) -> dict:
    raw_pys = {}
    for pp in py_files:
        rel = _rel_str(pp, root)
        if rel in g.modules: continue
        try: tree = ast.parse(pp.read_text(encoding="utf-8", errors="replace"))
        except SyntaxError: continue
        role, mod = _py_role_and_mod(rel, g.package)
        g.modules[rel] = ModuleNode(rel, str(pp.resolve()), role, pp.stem, "", [mod] if mod else [], [], 0)
        if mod: g.mod_to_nodes.setdefault(mod, []).append(rel)
        defs, imps = _py_defs_and_imports(tree)
        raw_pys[rel] = (tree, mod, role, imps)
        for nm, kind, ln, recv, _, bases, dlg in defs:
            _add_symbol(g, SymbolNode(f"{rel}::{nm}", nm, nm, kind, rel, mod, "py", "", "", ln, recv, bases, dlg))
    return raw_pys

Edge resolution propagates `_all_` re-exports, binds explicit and internal star imports, and emits resolved or ambiguous call edges.

In [ ]:
#| export
def _propagate_reexports(g: ProjectGraph, raw_cells: dict):
    for _, (nb, _, default_exp) in raw_cells.items():
        local_imp = {}
        for cell in nb.cells:
            if cell.get("cell_type") != "code" or not is_exported(cell): continue
            exp_mod = _qual_mod(g.package, cell_export_module(cell, default_exp) or "")
            tree, _ = parse_cell_ast(cell)
            if tree is None: continue
            for imp in cell_imports(cell):
                rm = _resolve_import_mod(imp.module, exp_mod)
                if not imp.is_star and rm in g.mod_exports and imp.name in g.mod_exports[rm]: local_imp[imp.alias] = g.mod_exports[rm][imp.name]
            for nm in _extract_reexports(tree):
                if nm in local_imp and exp_mod:
                    for uid in local_imp[nm]: g.mod_exports.setdefault(exp_mod, {}).setdefault(nm, []).append(uid)

def _bind_imports(g: ProjectGraph, rel: str, imps: list, cur_mod: str, cid: str, ch: str, is_lib: bool,
    bindings: dict, mod_aliases: dict, imp_adj: dict, add_edge):
    for mod, nm, alias, ln, is_star in imps:
        rm = _resolve_import_mod(mod, cur_mod)
        targets = g.mod_to_nodes.get(rm, []) or ([] if nm == "*" else g.mod_to_nodes.get(f"{rm}.{nm}", []))
        for tgt in targets:
            if is_lib: imp_adj[rel].add(tgt)
            add_edge(rel, tgt, "imports", "exact", rel, cid, ch, ln)
        if is_star and rm in g.mod_exports:
            for enm, uids in g.mod_exports[rm].items(): bindings[enm].extend(uids)
        elif rm in g.mod_exports and nm in g.mod_exports[rm]: bindings[alias].extend(g.mod_exports[rm][nm])
        elif f"{rm}.{nm}" in g.mod_exports: mod_aliases[alias] = f"{rm}.{nm}"
        elif rm in g.mod_exports and nm == rm: mod_aliases[alias] = rm

def _emit_calls(g: ProjectGraph, rel: str, tree, bindings: dict, mod_aliases: dict, owner_map: dict,
    cid: str, ch: str, is_test: bool, by_name: dict, patch_names: set, add_edge):
    ekind = "tests" if is_test else "calls"
    for owner, base, nm, ln in _ast_calls(tree):
        src_key = owner_map.get(owner, rel)
        if not base:
            for uid in bindings.get(nm, ()): add_edge(src_key, uid, ekind, "resolved", rel, cid, ch, ln)
        elif base in mod_aliases and nm in g.mod_exports.get(mod_aliases[base], {}):
            for uid in g.mod_exports[mod_aliases[base]][nm]: add_edge(src_key, uid, ekind, "resolved", rel, cid, ch, ln)
        elif nm in by_name and (nm in patch_names or nm not in _COMMON_ATTRS):
            for uid in by_name[nm]:
                if g.symbols[uid].node_id != rel or g.symbols[uid].cell_id != cid: add_edge(src_key, uid, ekind, "ambiguous", rel, cid, ch, ln)

In [ ]:
#| export
def _resolve_edges(g: ProjectGraph, raw_cells: dict, raw_pys: dict) -> dict:
    by_name, by_class, patch_names = defaultdict(list), defaultdict(list), set()
    for s in g.symbols.values():
        by_name[s.name].append(s.uid)
        if s.kind == "class": by_class[s.name].append(s.uid)
        elif s.kind == "patch": patch_names.add(s.name)
    _propagate_reexports(g, raw_cells)
    imp_adj, edge_seen = defaultdict(set), set()
    def _add_edge(src, dst, kind, conf, path="", cid="", ch="", ln=0):
        key = (src, dst, kind, cid, ln)
        if src == dst or key in edge_seen: return
        edge_seen.add(key)
        g.edges.append(GraphEdge(src, dst, kind, conf, path, cid, ch, ln))
    for rel, (nb, ch_by_idx, default_exp) in raw_cells.items():
        bindings, mod_aliases = defaultdict(list), {}
        for s in g.symbols.values():
            if s.node_id == rel: bindings[s.name].append(s.uid)
        for idx, cell in enumerate(nb.cells):
            if cell.get("cell_type") != "code": continue
            cid, ch = str(cell.get("id", f"cell-{idx}")), ch_by_idx.get(idx, "")
            tree, _ = parse_cell_ast(cell)
            if tree is None: continue
            exp = is_exported(cell)
            cur_mod = _qual_mod(g.package, cell_export_module(cell, default_exp) or default_exp or "")
            imps = [(i.module, i.name, i.alias, i.lineno, i.is_star) for i in cell_imports(cell)]
            _bind_imports(g, rel, imps, cur_mod, cid, ch, exp, bindings, mod_aliases, imp_adj, _add_edge)
            cell_syms = [s for s in g.symbols.values() if s.node_id == rel and s.cell_id == cid]
            _emit_calls(g, rel, tree, bindings, mod_aliases, {s.name: s.uid for s in cell_syms}, cid, ch, not exp, by_name, patch_names, _add_edge)
            for s in cell_syms:
                if s.kind == "patch" and s.receiver:
                    for cuid in bindings.get(s.receiver, ()) or by_class.get(s.receiver, ()):
                        _add_edge(s.uid, cuid, "patches", "exact" if cuid in bindings.get(s.receiver, ()) else "resolved", rel, cid, ch, s.lineno)
                for b in s.bases:
                    bname = b.split(".")[-1]
                    for buid in bindings.get(bname, ()) or by_class.get(bname, ()):
                        _add_edge(s.uid, buid, "inherits", "resolved" if buid in bindings.get(bname, ()) else "ambiguous", rel, cid, ch, s.lineno)
                if s.delegates_to:
                    dname = s.delegates_to.split(".")[-1]
                    for duid in bindings.get(dname, ()) or by_name.get(dname, ()):
                        _add_edge(s.uid, duid, "delegates", "resolved" if duid in bindings.get(dname, ()) else "ambiguous", rel, cid, ch, s.lineno)
        for ref, cid, ch in g.doc_refs.get(rel, ()):
            for uid in by_name.get(ref.split(".")[-1], ()): _add_edge(rel, uid, "docs_ref", "exact", rel, cid, ch, 0)
    for rel, (tree, mod, role, imps) in raw_pys.items():
        bindings, mod_aliases = defaultdict(list), {}
        py_owners = {s.name: s.uid for s in g.symbols.values() if s.node_id == rel}
        for nm, uid in py_owners.items(): bindings[nm].append(uid)
        _bind_imports(g, rel, imps, mod, "", "", role != "test", bindings, mod_aliases, imp_adj, _add_edge)
        _emit_calls(g, rel, tree, bindings, mod_aliases, py_owners, "", "", role == "test", by_name, patch_names, _add_edge)
    return imp_adj

`nb_graph` wires parsing, edge resolution, graph analytics, and `(mtime_ns, size)` caching into one entry point.

In [ ]:
#| export
def nb_graph(
    path: str | Path = ".",  # Project root, `nbs/` directory, or single notebook
    *,
    include_py: bool = True,  # Include handwritten `.py` files and `tests/`
    use_cache: bool = True,  # Reuse cached graph when file mtimes and sizes match
) -> ProjectGraph:
    "Build a deterministic hybrid structural graph of notebooks and handwritten Python files."
    root, nb_files, py_files = _resolve_scope(path)
    if not include_py: py_files = []
    sig = _cache_signature(root, nb_files, py_files, include_py)
    if use_cache and sig in _GRAPH_CACHE: return _GRAPH_CACHE[sig]
    g = ProjectGraph(str(root), package_name(root))
    raw_cells, raw_pys = _index_notebooks(g, root, nb_files), _index_py_files(g, root, py_files)
    imp_adj = _resolve_edges(g, raw_cells, raw_pys)
    lib_nodes = [nid for nid, m in g.modules.items() if m.role != "test"]
    sccs = _tarjan_sccs(lib_nodes, imp_adj)
    g.cycles = [c for c in sccs if len(c) > 1]
    layers, ranks = _compute_layers(lib_nodes, imp_adj, sccs), _compute_pagerank(lib_nodes, imp_adj)
    indeg = Counter(v for u in lib_nodes for v in imp_adj.get(u, ()) if v in g.modules)
    for nid, m in g.modules.items():
        m.layer, m.centrality, m.in_degree, m.out_degree = layers.get(nid, 0), ranks.get(nid, 0.0), indeg.get(nid, 0), len(imp_adj.get(nid, ()))
    nb_nodes = [nid for nid, m in g.modules.items() if m.role in ("library", "unexported_nb")]
    g.communities, g.surprising = _compute_communities(g, nb_nodes, imp_adj), _find_surprising(g, imp_adj)
    if use_cache: _GRAPH_CACHE[sig] = g
    return g

A small disposable hybrid project with two notebooks, a `@patch`, and a handwritten `tests/` file exercises import resolution, layers, hubs, and test edges.

In [ ]:
with TemporaryDirectory() as tmp:
    shop_root = Path(tmp)
    (shop_root / "pyproject.toml").write_text('[project]\nname = "shop"\n')
    for d in ("nbs", "shop", "tests"): (shop_root / d).mkdir()
    write_nb(new_nb([mk_cell("#| default_exp core"), mk_cell("#| export\nclass Money:\n    def __init__(self, x): self.x = x\ndef tax(x): return round(x * 0.19, 2)")]), shop_root / "nbs/00_core.ipynb")
    write_nb(new_nb([mk_cell("#| default_exp billing"), mk_cell("#| export\nfrom shop.core import Money, tax as calc_tax"), mk_cell("#| export\n@patch\ndef gross(self: Money): return self.x + calc_tax(self.x)")]), shop_root / "nbs/01_billing.ipynb")
    (shop_root / "tests/test_billing.py").write_text("from shop.core import tax\nassert tax(100) == 19.0\n")
    g = nb_graph(shop_root, use_cache=False)
    test_eq((g.modules["nbs/00_core.ipynb"].layer, g.modules["nbs/01_billing.ipynb"].layer, [p.qualname for p in g.patches_for("Money")]), (0, 1, ["Money.gross"]))
    assert any(e.kind == "calls" and e.confidence == "resolved" for e in g.edges) and any(e.kind == "tests" and e.src == "tests/test_billing.py" for e in g.edges)

## Architecture overview (`nb_arch`)

`nb_arch` compresses even a 90-notebook repository into a compact mesoscale report: dependency layers, God-Hubs (high fan-in), orchestrators/entrypoints (high fan-out), subsystem communities, unexported notebooks, and surprising connections.

In [ ]:
#| export
@dataclass
class ArchReport:
    "Compact mesoscale architecture summary for an nbdev project."
    root: str
    package: str | None
    counts: dict
    layers: dict
    hubs: list[ModuleNode]
    entrypoints: list[ModuleNode]
    communities: list[Community]
    unexported: list[str]
    surprising: list

    def __str__(self) -> str:
        c = self.counts
        lines = [f"Architecture {self.root} (pkg={self.package or '?'})",
            f"Files: {c['library']} library nbs, {c['unexported']} unexported nbs, {c['handwritten']} handwritten .py, {c['tests']} test .py | {c['symbols']} symbols, {c['edges']} edges"]
        if self.hubs:
            lines.append("\nGod Hubs (highest fan-in / centrality):")
            for m in self.hubs:
                mods = f" [{', '.join(m.export_modules)}]" if m.export_modules else ""
                lines.append(f"  - {m.id} (in={m.in_degree}, out={m.out_degree}, L{m.layer}): {m.title}{mods}")
        if self.entrypoints:
            lines.append("\nEntrypoints & Facades (highest fan-out):")
            for m in self.entrypoints: lines.append(f"  - {m.id} (out={m.out_degree}, in={m.in_degree}, L{m.layer}): {m.title}")
        if self.communities:
            multi = [cm for cm in self.communities if len(cm.nodes) > 1]
            shown = multi[:10] if multi else self.communities[:10]
            solo = len(self.communities) - len(multi)
            lines.append(f"\nSubsystem Communities ({len(shown)} shown{f', {solo} standalone' if multi and solo else ''}):")
            for cm in shown:
                syms = f" | key: {', '.join(cm.key_symbols[:4])}" if cm.key_symbols else ""
                lines.append(f"  - [{cm.id}] {cm.name} ({len(cm.nodes)} nbs, hub={Path(cm.hub).name}){syms}")
        if self.layers:
            lines.append("\nDependency Layers (foundation -> top):")
            for lvl in sorted(self.layers):
                names = [Path(n).name for n in self.layers[lvl]]
                lines.append(f"  L{lvl}: " + ", ".join(names[:8]) + (f", ... (+{len(names)-8})" if len(names) > 8 else ""))
        if self.unexported: lines.append(f"\nUnexported notebooks ({len(self.unexported)}): " + ", ".join(self.unexported[:8]) + (f", ... (+{len(self.unexported)-8})" if len(self.unexported) > 8 else ""))
        if self.surprising: lines += [f"\nSurprising Connections ({len(self.surprising)}):", *(f"  - {a} -> {b}: {r}" for a, b, r in self.surprising[:10])]
        return "\n".join(lines)

    def _repr_markdown_(self) -> str: return f"```text\n{self}\n```"

In [ ]:
#| export
def nb_arch(
    path: str | Path = ".",  # Project root or notebooks directory
    *,
    include_py: bool = True,  # Include handwritten `.py` files and `tests/`
    max_hubs: int = 8,  # Maximum hubs and entrypoints to display
) -> ArchReport:
    "Return a compact mesoscale architecture report: hubs, layers, communities, and surprising links."
    g = nb_graph(path, include_py=include_py)
    mods = list(g.modules.values())
    counts = dict(library=sum(m.role == "library" for m in mods), unexported=sum(m.role == "unexported_nb" for m in mods),
        handwritten=sum(m.role in ("handwritten_py", "script") for m in mods), tests=sum(m.role == "test" for m in mods),
        symbols=len(g.symbols), edges=len(g.edges))
    lib_mods = [m for m in mods if m.role in ("library", "handwritten_py", "script")]
    hubs = sorted([m for m in lib_mods if m.in_degree > 0], key=lambda m: (-m.in_degree, -m.centrality, m.id))[:max_hubs]
    entry = sorted([m for m in lib_mods if m.out_degree > 0], key=lambda m: (-m.out_degree, -m.layer, m.id))[:max_hubs]
    layers = defaultdict(list)
    nb_lib = [m for m in lib_mods if m.role == "library"]
    for m in sorted(nb_lib or lib_mods, key=lambda m: (m.layer, -m.in_degree, m.id)): layers[m.layer].append(m.id)
    unexp = sorted(m.id for m in mods if m.role == "unexported_nb")
    return ArchReport(g.root, g.package, counts, dict(layers), hubs, entry, g.communities, unexp, g.surprising)

Running `nb_arch` on a layered project surfaces the foundation hub, top-level facade, and layer inversion in one compact view.

In [ ]:
with TemporaryDirectory() as tmp:
    demo_root = Path(tmp)
    (demo_root / "pyproject.toml").write_text('[project]\nname = "demo"\n')
    (demo_root / "nbs").mkdir()
    write_nb(new_nb([mk_cell("#| default_exp core"), mk_cell("# Core", "markdown"), mk_cell("> Base.", "markdown"),
        mk_cell("#| export\nfrom demo.later import helper\ndef base(): return helper()")]), demo_root / "nbs/00_core.ipynb")
    write_nb(new_nb([mk_cell("#| default_exp later"), mk_cell("# Later", "markdown"), mk_cell("> Utility.", "markdown"),
        mk_cell("#| export\ndef helper(): return 1")]), demo_root / "nbs/05_later.ipynb")
    arch = nb_arch(demo_root)
    test_eq((arch.counts["library"], len(arch.surprising)), (2, 1))
    assert "layer inversion" in str(arch)
arch

## Blast radius (`impact`) and path tracing (`trace_path`)

Before changing a core function, chapter, or notebook, `impact` walks reverse dependencies transitively across notebooks, handwritten Python modules, and `tests/` files — and suggests the minimal `nbdev-test` and `pytest` commands to verify the change. `trace_path` finds the shortest dependency or call chain between any two modules or symbols.

In [ ]:
#| export
@dataclass
class ImpactHit:
    "One dependent symbol, notebook, or Python file in a blast radius."
    target: str
    path: str
    role: str
    hop: int
    via: str  # Edge kind ('calls', 'imports', 'patches', 'inherits', 'delegates', 'tests')
    confidence: str
    chapter: str = ""
    cell_id: str = ""

@dataclass
class ImpactReport:
    "Transitive blast radius and minimal verification plan for a symbol, chapter, or notebook."
    query: str
    seeds: list[str]
    hits: list[ImpactHit] = field(default_factory=list)
    notebooks: list[str] = field(default_factory=list)
    handwritten_py: list[str] = field(default_factory=list)
    test_files: list[str] = field(default_factory=list)
    suggested_commands: list[str] = field(default_factory=list)

    def __str__(self) -> str:
        lines = [f"Impact of {self.query} (seeds={len(self.seeds)}, impacted={len(self.hits)})"]
        if not self.hits: lines.append("No static dependents found.")
        else:
            for h in self.hits[:25]:
                loc = f"{h.path}#{h.cell_id}" if h.cell_id else h.path
                ch = f" [{h.chapter}]" if h.chapter else ""
                lines.append(f"  hop={h.hop} ({h.via}/{h.confidence}) {h.target} @ {loc}{ch}")
            if len(self.hits) > 25: lines.append(f"  ... (+{len(self.hits)-25} more)")
        if self.notebooks: lines.append(f"\nAffected notebooks ({len(self.notebooks)}): {', '.join(self.notebooks)}")
        if self.handwritten_py: lines.append(f"Affected handwritten .py ({len(self.handwritten_py)}): {', '.join(self.handwritten_py)}")
        if self.test_files: lines.append(f"Affected tests ({len(self.test_files)}): {', '.join(self.test_files)}")
        if self.suggested_commands: lines += ["\nMinimal verification commands:", *(f"  $ {cmd}" for cmd in self.suggested_commands)]
        return "\n".join(lines)

    def _repr_markdown_(self) -> str: return f"```text\n{self}\n```"

`impact` resolves symbols, chapters, or notebooks and walks reverse edges up to `depth` hops.

In [ ]:
#| export
def _resolve_target_ids(g: ProjectGraph, query: str) -> list[str]:
    q = query.strip()
    if "#" in q:
        nb_part, sub = q.rsplit("#", 1)
        nb_matches = [nid for nid in g.modules if nid == nb_part or Path(nid).name == nb_part or nid.endswith(nb_part)]
        if nb_matches:
            syms = [s.uid for s in g.symbols.values() if s.node_id in nb_matches and (s.name == sub or s.qualname == sub or sub.lower() in s.chapter.lower())]
            if syms: return syms
    if q in g.modules: return [q]
    nb_exact = [nid for nid in g.modules if Path(nid).name == q or Path(nid).stem == q or nid.endswith("/" + q)]
    if nb_exact: return nb_exact
    if q in g.mod_to_nodes: return list(g.mod_to_nodes[q])
    if g.package and f"{g.package}.{q}" in g.mod_to_nodes: return list(g.mod_to_nodes[f"{g.package}.{q}"])
    syms = [s.uid for s in g.symbols_named(q)]
    if syms: return syms
    pool = sorted(set(g.modules) | {Path(n).stem for n in g.modules} | {s.name for s in g.symbols.values()})
    close = difflib.get_close_matches(q, pool, n=5)
    hint = f" Closest: {', '.join(close)}" if close else ""
    raise ValueError(f"Target {query!r} not found in graph.{hint}")

def impact(
    target: str,  # Symbol (`tax`, `Money.gross`), notebook (`00_core.ipynb`), or `nb.ipynb#Chapter`
    scope: str | Path = ".",  # Project root or directory
    *,
    depth: int = 3,  # Maximum transitive hops
    include_ambiguous: bool = False,  # Include name-only ambiguous matches
) -> ImpactReport:
    "Trace direct and transitive dependents of `target` across notebooks, handwritten `.py`, and tests."
    g = nb_graph(scope)
    seeds = _resolve_target_ids(g, target)
    seed_set = set(seeds)
    for sid in list(seeds):
        if sid in g.modules: seed_set.update(s.uid for s in g.symbols.values() if s.node_id == sid and s.module)
    rev = defaultdict(list)
    for e in g.edges:
        if e.kind != "docs_ref" and (include_ambiguous or e.confidence != "ambiguous"): rev[e.dst].append(e)
    visited, q, hits = set(seed_set), deque((s, 0) for s in seed_set), []
    while q:
        cur, hop = q.popleft()
        if hop >= depth: continue
        for e in rev.get(cur, ()):
            if e.src in visited: continue
            visited.add(e.src)
            sym = g.symbols.get(e.src)
            nid = sym.node_id if sym else e.src
            role = g.modules[nid].role if nid in g.modules else "unknown"
            ch, cid = e.chapter or (sym.chapter if sym else ""), e.cell_id or (sym.cell_id if sym else "")
            hits.append(ImpactHit(sym.qualname if sym else e.src, nid, role, hop + 1, e.kind, e.confidence, ch, cid))
            q.append((e.src, hop + 1))
            if sym and nid not in visited: q.append((nid, hop + 1))
    hits.sort(key=lambda h: (h.hop, h.role == "test", h.path, h.target))
    seed_nbs = {g.symbols[s].node_id if s in g.symbols else s for s in seeds if (g.symbols[s].node_id if s in g.symbols else s).endswith(".ipynb")}
    nbs = sorted(seed_nbs | {h.path for h in hits if h.path.endswith(".ipynb")})
    hand, tests = sorted({h.path for h in hits if h.role in ("handwritten_py", "script")}), sorted({h.path for h in hits if h.role == "test"})
    cmds = ([f"nbdev-test --file-re '({'|'.join(re.escape(Path(n).name) for n in nbs)})'"] if nbs else []) + (["pytest -q " + " ".join(tests[:10])] if tests else [])
    return ImpactReport(target, sorted(seeds), hits, nbs, hand, tests, cmds)

`trace_path` runs a shortest-path search over module and symbol edges to explain how two parts of a codebase connect.

In [ ]:
#| export
@dataclass
class PathTrace:
    "Shortest dependency or call path between two graph nodes."
    src: str
    dst: str
    edges: list[GraphEdge] = field(default_factory=list)

    @property
    def found(self) -> bool: return bool(self.edges)

    def __str__(self) -> str:
        if not self.edges: return f"No path from {self.src} to {self.dst}"
        lines = [f"Path {self.src} -> {self.dst} ({len(self.edges)} hop(s)):"]
        for i, e in enumerate(self.edges, 1):
            loc = f" @ {e.path}#{e.cell_id}:{e.lineno}" if e.cell_id else (f" @ {e.path}:{e.lineno}" if e.path and e.lineno else "")
            lines.append(f"  {i}. {e.src} --[{e.kind}/{e.confidence}]--> {e.dst}{loc}")
        return "\n".join(lines)

    def _repr_markdown_(self) -> str: return f"```text\n{self}\n```"

def trace_path(
    src: str,  # Starting notebook, module, or symbol
    dst: str,  # Target notebook, module, or symbol
    scope: str | Path = ".",  # Project root or directory
    *,
    max_depth: int = 6,  # Maximum hops to search
) -> PathTrace:
    "Find the shortest structural dependency or call path from `src` to `dst`."
    g = nb_graph(scope)
    starts, goals = set(_resolve_target_ids(g, src)), set(_resolve_target_ids(g, dst))
    adj = defaultdict(list)
    for e in g.edges:
        if e.kind != "docs_ref": adj[e.src].append(e)
    for s in g.symbols.values():
        adj[s.node_id].append(GraphEdge(s.node_id, s.uid, "defines", "exact", s.node_id, s.cell_id, s.chapter, s.lineno))
        adj[s.uid].append(GraphEdge(s.uid, s.node_id, "defined_in", "exact", s.node_id, s.cell_id, s.chapter, s.lineno))
    q, visited = deque((s, []) for s in sorted(starts)), set(starts)
    while q:
        cur, trail = q.popleft()
        if cur in goals and trail: return PathTrace(src, dst, trail)
        if len(trail) >= max_depth: continue
        for e in adj.get(cur, ()):
            if e.dst in visited: continue
            visited.add(e.dst)
            q.append((e.dst, trail + [e]))
    return PathTrace(src, dst, [])

Tracing impact and shortest path on a three-layer project shows transitive notebook dependents, handwritten `tests/*.py` coverage, and the exact verification commands.

In [ ]:
with TemporaryDirectory() as tmp:
    erp_root = Path(tmp)
    (erp_root / "pyproject.toml").write_text('[project]\nname = "erp"\n')
    for d in ("nbs", "erp", "tests"): (erp_root / d).mkdir()
    write_nb(new_nb([mk_cell("#| default_exp core"), mk_cell("#| export\ndef rate(): return 0.19")]), erp_root / "nbs/00_core.ipynb")
    write_nb(new_nb([mk_cell("#| default_exp calc"), mk_cell("#| export\nfrom erp.core import rate"), mk_cell("#| export\ndef vat(x): return x * rate()")]), erp_root / "nbs/01_calc.ipynb")
    write_nb(new_nb([mk_cell("#| default_exp api"), mk_cell("#| export\nfrom erp.calc import vat"), mk_cell("#| export\ndef invoice(x): return x + vat(x)")]), erp_root / "nbs/02_api.ipynb")
    (erp_root / "tests/test_api.py").write_text("from erp.api import invoice\nassert invoice(100) == 119.0\n")
    imp, tr = impact("rate", scope=erp_root), trace_path("invoice", "rate", scope=erp_root)
    test_eq((len(imp.notebooks), imp.test_files, tr.found, len(tr.edges)), (3, ["tests/test_api.py"], True, 2))